# Example-05: Default twiss (__call__ method)

In [1]:
# Import

import numpy
import pandas
import torch

import sys
sys.path.append('..')

from harmonica.util import mod
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter
from harmonica.decomposition import Decomposition
from harmonica.model import Model
from harmonica.table import Table
from harmonica.twiss import Twiss

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())

True


In [2]:
# Set data type and device

dtype = torch.float64
device = 'cpu'

In [3]:
# Compute reference parameters (frequency, amplitude and phase)

df = pandas.read_pickle('../virtual_tbt.pkl.gz')

length = 4096
w = Window(length, 'cosine_window', 4.0, dtype=dtype, device=device)

x = Data.from_data(w, torch.tensor(df.X.to_list(), dtype=dtype, device=device))
y = Data.from_data(w, torch.tensor(df.Y.to_list(), dtype=dtype, device=device))

f = Frequency(x)
x.window_remove_mean()
x.window_apply()
f('parabola')
x.reset()
ref_nux, ref_sigma_nux = 1.0 - f.frequency.mean(), f.frequency.std()
d = Decomposition(x)
result, _ = d.harmonic_sum(ref_nux, w.window, x.data)
_, _, ref_ax, ref_fx = result.T

f = Frequency(y)
y.window_remove_mean()
y.window_apply()
f('parabola')
y.reset()
ref_nuy, ref_sigma_nuy = 1.0 - f.frequency.mean(), f.frequency.std()
d = Decomposition(y)
result, _ = d.harmonic_sum(ref_nuy, w.window, y.data)
_, _, ref_ay, ref_fy = result.T

In [4]:
# Set noise

noise_x = 1.0E-6*(25.0 + 25.0*torch.rand(x.size, dtype=dtype, device=device))
noise_y = 1.0E-6*(25.0 + 25.0*torch.rand(y.size, dtype=dtype, device=device))

In [5]:
# Compute frequency, amplitude and phase for x plane with noise

length = 1024
w = Window(length, 'cosine_window', 1.0, dtype=dtype, device=device)
d = Data.from_data(w, x.data[:, :length])
d.add_noise(noise_x)
d.data.copy_(d.work)

# Frequency

f = Frequency(d)
d.window_remove_mean()
d.window_apply()
f('parabola')
d.reset()
nux, sigma_nux = 1.0 - f.frequency.mean(), f.frequency.std()
print(f'nux={nux.item():12.9}, sigma_nux={sigma_nux.item():12.9}, error_nux={abs(ref_nux - nux).item():12.9}')
print()

# Amplitude & phase

d = Decomposition(d)

ax, sigma_ax, _ = d.harmonic_amplitude(nux, length=64, order=1.0, error=True, sigma_frequency=sigma_nux, shift=True, count=64, step=8, method='noise')
print(f'{(ax - ref_ax).abs().sum().item()=:12.9}')
print()

fx, sigma_fx, _ = d.harmonic_phase(nux, length=256, order=0.0, error=True, sigma_frequency=sigma_nux, shift=True, count=256, step=8, method='noise')
print(f'{(fx - ref_fx).abs().sum().item()=:12.9}')
print()

nux= 0.536883233, sigma_nux=6.24883135e-07, error_nux=1.34179748e-07

(ax - ref_ax).abs().sum().item()=9.26938067e-05

(fx - ref_fx).abs().sum().item()=0.0437316705



In [6]:
# Compute frequency, amplitude and phase for y plane with noise

length = 1024
w = Window(length, 'cosine_window', 1.0, dtype=dtype, device=device)
d = Data.from_data(w, y.data[:, :length])
d.add_noise(noise_y)
d.data.copy_(d.work)

# Frequency

f = Frequency(d)
d.window_remove_mean()
d.window_apply()
f('parabola')
d.reset()
nuy, sigma_nuy = 1.0 - f.frequency.mean(), f.frequency.std()
print(f'nuy={nuy.item():12.9}, sigma_nuy={sigma_nuy.item():12.9}, error_nuy={abs(ref_nuy - nuy).item():12.9}')
print()

# Amplitude & phase

d = Decomposition(d)

ay, sigma_ay, _ = d.harmonic_amplitude(nuy, length=64, order=1.0, error=True, sigma_frequency=sigma_nuy, shift=True, count=64, step=8, method='noise')
print(f'{(ay - ref_ay).abs().sum().item()=:12.9}')
print()

fy, sigma_fy, _ = d.harmonic_phase(nuy, length=256, order=0.0, error=True, sigma_frequency=sigma_nuy, shift=True, count=256, step=8, method='noise')
print(f'{(fy - ref_fy).abs().sum().item()=:12.9}')
print()

nuy= 0.576774928, sigma_nuy=9.43039331e-07, error_nuy=2.94495463e-07

(ay - ref_ay).abs().sum().item()=8.7926045e-05

(fy - ref_fy).abs().sum().item()= 0.050324919



In [7]:
# Set model & table

model = Model(path='../config.yaml', dtype=dtype, device=device)
table = Table([name for name, kind in zip(model.name, model.kind) if kind == 'MONITOR'], nux, nuy, ax, ay, fx, fy, sigma_nux, sigma_nuy, sigma_ax, sigma_ay, sigma_fx, sigma_fy, dtype=dtype, device=device)

In [8]:
# Default twiss (compute action, beta from amplitude, virtual phase, corrected phase and twiss from phase)

twiss = Twiss(model, table, limit=8)
twiss()

,name,kind,flag,time,ax,sigma_ax,bx,sigma_bx,fx,sigma_fx,ay,sigma_ay,by,sigma_by,fy,sigma_fy
0,HEAD,VIRTUAL,0,0.000000,-0.678870,0.001783,7.470305,0.018109,0.595534,0.000858,0.654076,0.000845,15.846072,0.014515,-0.580410,0.000436
1,STP2,MONITOR,1,0.000000,-0.680424,0.001931,7.469018,0.019870,0.596644,0.004405,0.654355,0.000776,15.847460,0.014278,-0.580463,0.005043
2,IV4P,VIRTUAL,0,4.153500,1.954952,0.005259,14.239014,0.036473,0.981376,0.000865,-2.759814,0.003116,13.854139,0.014579,-0.276708,0.000393
3,STP4,MONITOR,1,6.667000,0.383599,0.001777,3.121721,0.008660,1.431949,0.004528,3.090910,0.002802,30.430285,0.030097,-0.173068,0.004989
4,SRP1,MONITOR,1,10.190565,-0.802175,0.002126,4.734830,0.009744,2.561266,0.004486,1.840426,0.001574,13.266632,0.013139,0.001367,0.005320
5,SRP2,MONITOR,1,17.337624,-0.717959,0.002698,4.560172,0.014271,-2.624830,0.004469,1.840290,0.002164,13.278778,0.017645,0.794755,0.005176
6,SRP3,MONITOR,1,24.484683,-0.768728,0.001412,4.483756,0.008701,-1.466850,0.004547,1.847040,0.002814,13.324960,0.018634,1.586839,0.005238
7,SRP4,MONITOR,1,31.631742,-0.789470,0.002063,4.755588,0.010054,-0.353836,0.004440,1.848829,0.002350,13.305853,0.015778,2.377236,0.005493
8,SRP5,MONITOR,1,38.778801,-0.714419,0.002237,4.490434,0.014608,0.755318,0.004793,1.838976,0.003249,13.263850,0.026063,-3.111887,0.005085
9,SRP6,MONITOR,1,45.925860,-0.787742,0.001840,4.551860,0.010330,1.912952,0.004386,1.837712,0.002543,13.275030,0.017108,-2.317925,0.005388


In [9]:
# Get twiss parameter by index/name

print(twiss.get_ax(28))
print(twiss.get_ax('IP'))

tensor([-1.856490861459e-04, 5.831309446297e-04], dtype=torch.float64)
tensor([-1.856490861459e-04, 5.831309446297e-04], dtype=torch.float64)


In [10]:
# Get all twiss parameters by index/name

for key, value in twiss.get_twiss(28).items():
    print(f'{key:12.9}: {value.item():12.9}')
else:
    print()
    
for key, value in twiss.get_twiss('IP').items():
    print(f'{key:12.9}: {value.item():12.9}')
else:
    print()

ax          : -0.000185649086
sigma_ax    : 0.000583130945
bx          :  0.750483116
sigma_bx    : 0.00113400343
fx          :   1.01979752
sigma_fx    : 0.000514129881
ay          : 0.000112760537
sigma_ay    : 0.000954448576
by          : 0.0677600865
sigma_by    : 0.000147897465
fy          :    3.0990659
sigma_fy    : 0.000858994623

ax          : -0.000185649086
sigma_ax    : 0.000583130945
bx          :  0.750483116
sigma_bx    : 0.00113400343
fx          :   1.01979752
sigma_fx    : 0.000514129881
ay          : 0.000112760537
sigma_ay    : 0.000954448576
by          : 0.0677600865
sigma_by    : 0.000147897465
fy          :    3.0990659
sigma_fy    : 0.000858994623

